In [ ]:
import os
import re
import json
from datetime import datetime
import requests

# 0. KONFIGURATION

OPENROUTER_API_KEY = ""   


LLM_MODEL = "openai/gpt-4.1-mini"

# Text-Eingabe (fertiges Transkript mit Sprecherlabels)
INPUT_TEXT = "SynthText.txt"

# Optional: Ground-Truth-JSON zum Vergleichen
GROUNDTRUTH_PATH = "groundtruth.json"  # kann leer/nicht vorhanden sein


# ==========================
# 1. HILFSFUNKTIONEN
# ==========================

def normalize_speakers(text: str) -> str:
    """
    Stellt sicher, dass Sprecherlabels am Anfang von Zeilen stehen.
    Z.B. 'Sprecher 1:', 'Sprecher 2:', 'Spielleiter:', 'Rot:', 'Blau:'.
    """
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    patterns = [
        r"(Sprecher\s+\d+:)",
        r"(Spielleiter:)",
        r"(Rot:)",
        r"(Blau:)",
    ]

    for p in patterns:
        text = re.sub(r"\s*" + p, r"\n\1", text)

    lines = [line.rstrip() for line in text.split("\n")]
    while lines and lines[0] == "":
        lines.pop(0)
    while lines and lines[-1] == "":
        lines.pop()
    return "\n".join(lines)


def extract_json_from_text(text: str) -> str:
    """Extrahiert den JSON-Block aus einem LLM-Output."""
    match = re.search(r"\{[\s\S]*\}", text)
    if not match:
        raise ValueError("Kein JSON im Modelloutput gefunden.")
    return match.group(0)


def call_llm_for_analysis(transcript: str) -> dict:
    """
    Schickt das fertige Transkript (mit Sprecherlabels) an das LLM
    und erhält eine strukturierte Auswertung:
    - Teilnehmer
    - Runden & Phasen
    - Würfe
    - T-Shirt-Größen
    - Argumente
    - Lücken (Inhalte die nicht in den jeweiligen Argumenten auftauchen)
    - Summary für den OnePager
    - Konkrete Empfehlungen / Verbesserungen basierend auf dem Szenario
    """
    RULES_TEXT = """
Op4C Wargame (Kurzfassung):

- Zwei Teams: Rot (Angreifer, z.B. Clara), Blau (Verteidiger, z.B. BWI / Antonio).
- 2 Runden, jede Runde:
  - Phase 1: Intelphase
    * Blau würfelt (3W6, >=12 -> kennt Intel-Vorteil von Rot)
    * Rot würfelt Intel-Vorteil (Bonus 0/+1/+2 und Angriffsart: Cyber/Personal/Sabotage)
  - Phase 2: Diskussionsgefecht mit 4 Dimensionen:

Dimensionen (T-Shirt-Größen S,M,L,XL):
- Ressourcen
- Angriffskomplexität
- Verteidigung
- Aktionszeitfenster

Am Ende jeder Runde: Erfolgswurf Rot (3W6 >= Summe der 4 Dimensionen) -> Auswirkung.
"""

    INSTRUCTIONS = """
Analysiere das folgende Transkript einer Op4C-Sitzung.

GIB MIR AUSSCHLIESSLICH EIN JSON-OBJEKT MIT FOLGENDER STRUKTUR ZURÜCK:

{
  "participants": {
    "rot": "<Name oder 'Rot'>",
    "blau": "<Name oder 'Blau'>"
  },
  "rounds": [
    {
      "round": 1,
      "phase1": {
        "intel_blau_kennt_rot": true/false,
        "intel_vorteil_rot_bonus": 0/1/2/null,
        "intel_vorteil_rot_typ": "cyber"|"personal"|"sabotage"|null,
        "wuerfe_blau": "<Beschreibung oder null>",
        "wuerfe_rot": "<Beschreibung oder null>"
      },
      "phase2": {
        "ressourcen": "S"|"M"|"L"|"XL"|null,
        "komplexitaet": "S"|"M"|"L"|"XL"|null,
        "verteidigung": "S"|"M"|"L"|"XL"|null,
        "aktionszeitfenster": "S"|"M"|"L"|"XL"|null,
        "tabellensumme": <Zahl oder null>,
        "erfolgswurf_rot": "erfolgreich"|"nicht ausreichend"|"nicht erfolgreich"|null,
        "auswirkung": "voller erfolg"|"teilerfolg"|"kein erfolg"|null,
        "argumente": {
          "ressourcen": "<wichtigste Argumente oder ''>",
          "komplexitaet": "<...>",
          "verteidigung": "<...>",
          "aktionszeitfenster": "<...>"
        },
        "luecken": "<wo wurden Dinge nicht geklärt / diskutiert?>"
      }
    }
    // weitere Runden analog
  ],
  "summary": {
    "szenario": "<kurze Beschreibung des Szenarios (z.B. wer greift wen an, welches Ziel?)>",
    "angriffsergebnisse": "<kurze Zusammenfassung der Angriffe und Ergebnisse>",
    "verteidigungsmassnahmen_bwi": "<wichtigste Verteidigungsmaßnahmen von Blau/BWI>",
    "angriffsverlauf": "<wie hat sich der Angriff über die Runden entwickelt>",
    "schwachstellen": "<welche Schwachstellen wurden genannt?>",
    "luecken": "<welche Lücken/Unsicherheiten bleiben offen?>",
    "verbesserungsvorschlaege": "<konkrete Empfehlungen, was an Verteidigung, Prozessen, Technik, Awareness verbessert werden sollte>",
    "empfohlene_naechste_schritte": "<2–5 priorisierte nächste Schritte für die Organisation in diesem Szenario>"
  }
}

REGELN:
- Nutze nur Infos aus dem Transkript (keine Fantasie).
- Empfehlungen müssen logisch aus den im Transkript genannten Schwachstellen / Lücken abgeleitet sein.
- Wenn etwas im Transkript nicht vorkommt: nutze null oder "".
- KEINE Erklärtexte außerhalb des JSON. NUR das JSON-Objekt zurückgeben.
"""

    system_text = f"Du bist eine KI, die Op4C-Wargaming-Transkripte analysiert.\n{RULES_TEXT}"

    payload = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": system_text},
            {
                "role": "user",
                "content": INSTRUCTIONS
                          + "\n\n=== TRANSKRIPT BEGINN ===\n"
                          + transcript
                          + "\n=== TRANSKRIPT ENDE ==="
            },
        ],
        "stream": False,
    }

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }

    resp = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers=headers,
        data=json.dumps(payload),
    )
    resp.raise_for_status()
    content = resp.json()["choices"][0]["message"]["content"]

    json_text = extract_json_from_text(content)
    return json.loads(json_text)


def evaluate_against_groundtruth(pred: dict, gt: dict) -> dict:
    """
    Vergleicht Modell-Output mit GroundTruth:
    - T-Shirt-Größen (4 Dimensionen)
    - Erfolgswurf Rot
    """
    dims = ["ressourcen", "komplexitaet", "verteidigung", "aktionszeitfenster"]

    pred_rounds = pred.get("rounds", [])
    gt_rounds = gt.get("rounds", [])

    n = min(len(pred_rounds), len(gt_rounds))

    correct_tshirts = 0
    total_tshirts = 0
    correct_erfolgswurf = 0
    total_erfolgswurf = 0

    for i in range(n):
        pr2 = pred_rounds[i].get("phase2", {})
        gt2 = gt_rounds[i].get("phase2", {})

        for d in dims:
            if gt2.get(d) is None:
                continue
            total_tshirts += 1
            if pr2.get(d) == gt2.get(d):
                correct_tshirts += 1

        if gt2.get("erfolgswurf_rot") is not None:
            total_erfolgswurf += 1
            if pr2.get("erfolgswurf_rot") == gt2.get("erfolgswurf_rot"):
                correct_erfolgswurf += 1

    return {
        "tshirt_accuracy": correct_tshirts / total_tshirts if total_tshirts > 0 else None,
        "erfolgswurf_accuracy": correct_erfolgswurf / total_erfolgswurf if total_erfolgswurf > 0 else None,
    }


def build_onepager(data: dict) -> str:
    summary = data.get("summary", {})
    participants = data.get("participants", {})

    onepager = f"""
=== OP4C ONE-PAGER ===

Teilnehmer:
- Rot: {participants.get('rot', 'Unbekannt')}
- Blau: {participants.get('blau', 'Unbekannt')}

0. Szenario
{summary.get('szenario', '–')}

1. Angriffsergebnisse
{summary.get('angriffsergebnisse', '–')}

2. Verteidigungsmaßnahmen (BWI / Blau)
{summary.get('verteidigungsmassnahmen_bwi', '–')}

3. Angriffsverlauf
{summary.get('angriffsverlauf', '–')}

4. Schwachstellen
{summary.get('schwachstellen', '–')}

5. Lücken / Unsicherheiten
{summary.get('luecken', '–')}

6. Empfehlungen (übergeordnet)
{summary.get('verbesserungsvorschlaege', '–')}

7. Empfohlene nächste Schritte (priorisiert)
{summary.get('empfohlene_naechste_schritte', '–')}
""".strip()

    return onepager


# ==========================
# 2. PIPELINE: TXT → JSON → ONEPAGER
# ==========================

def main():
    if not os.path.exists(INPUT_TEXT):
        raise FileNotFoundError(f"Eingabetext nicht gefunden: {INPUT_TEXT}")

    # 1) TXT einlesen (anstatt Audio+Whisper)
    with open(INPUT_TEXT, "r", encoding="utf-8") as f:
        raw_txt = f.read()

    # 2) Sprecherformat ggf. noch etwas glätten
    transcript = normalize_speakers(raw_txt)

    # 3) LLM-Analyse → strukturierte JSON-Daten
    print("[Analyse] Sende Transkript an LLM...")
    extracted = call_llm_for_analysis(transcript)

    run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_json = f"runde_{run_id}_data.json"
    out_onepager = f"runde_{run_id}_onepager.txt"

    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(extracted, f, indent=2, ensure_ascii=False)
    print(f"[Analyse] Strukturierte Daten gespeichert in: {out_json}")

    # 4) Optional: Ground-Truth-Vergleich (wenn vorhanden)
    if os.path.exists(GROUNDTRUTH_PATH):
        with open(GROUNDTRUTH_PATH, "r", encoding="utf-8") as f:
            groundtruth = json.load(f)
        metrics = evaluate_against_groundtruth(extracted, groundtruth)
        print("[Eval] Accuracy vs. Ground Truth:", metrics)
    else:
        print("[Eval] Keine groundtruth.json gefunden – Accuracy-Scheck übersprungen.")

    # 5) OnePager bauen
    onepager = build_onepager(extracted)
    with open(out_onepager, "w", encoding="utf-8") as f:
        f.write(onepager)
    print(f"[OnePager] gespeichert in: {out_onepager}")
    print("\n=== OnePager Vorschau ===\n")
    print(onepager)


if __name__ == "__main__":
    main()


[Analyse] Sende Transkript an LLM...
[Analyse] Strukturierte Daten gespeichert in: runde_20251126_114127_data.json
[Eval] Accuracy vs. Ground Truth: {'tshirt_accuracy': 0.25, 'erfolgswurf_accuracy': 0.5}
[OnePager] gespeichert in: runde_20251126_114127_onepager.txt

=== OnePager Vorschau ===

=== OP4C ONE-PAGER ===

Teilnehmer:
- Rot: Lea
- Blau: Markus

0. Szenario
Team Rot (Lea) versucht, kritische Infrastruktur einer fiktiven Stadt durch Sabotage- und Personalangriffe lahmzulegen, um politische Reaktionen zu erzwingen. Team Blau (Markus) verteidigt Energieversorgung und Stadtverwaltung.

1. Angriffsergebnisse
Runde 1: Sabotage der Stromversorgung erfolgreich, langanhaltender Stromausfall mit erheblichen Auswirkungen. Runde 2: Personalangriff erfolgreich, Kommunikationsstörungen im Schlüsselpersonal verzögern Krisenmanagement.

2. Verteidigungsmaßnahmen (BWI / Blau)
Strenge Zugangskontrollen und Netzwerkmonitoring beim Energieversorger; Schulungen und Zugangskonzepte bei Stadtverwalt